## DagsHub for MLFlow

In [ ]:
import dagshub

dagshub.init(repo_owner="cykanp", repo_name="zalo-photos-assignment-2026", mlflow=True)

Accessing as cykanp


Initialized MLflow to track repo "cykanp/zalo-photos-assignment-2026"


Repository cykanp/zalo-photos-assignment-2026 initialized!


In [ ]:
# --- SETUP FOLDS ---
from datetime import datetime

import fiftyone as fo
import mlflow
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments

from z_photos.datasets import CanonicalLabel

labels_list = [c.value for c in CanonicalLabel]
label2id = {lbl: i for i, lbl in enumerate(labels_list)}
id2label = dict(enumerate(labels_list))

dataset = fo.load_dataset("z_photos-gold-train")
view = dataset.match(
    (fo.ViewField("keep_for_train")) & (fo.ViewField("canonical_label").exists())
)
labels = view.values("canonical_label")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
dataset.set_values("fold_id", [None] * len(dataset))
fold_ids = [None] * len(view)
for fold_idx, (_train_idx, val_idx) in enumerate(
    skf.split(np.zeros(len(labels)), labels)
):
    for vi in val_idx:
        fold_ids[vi] = fold_idx
view.set_values("fold_id", fold_ids)
dataset.save()

print(f"Total train samples: {len(view)}")
print(f"Folds assigned: {view.count_values('fold_id')}")

Total train samples: 240
Folds assigned: {1: 48, 0: 48, 4: 48, 3: 48, 2: 48}


In [ ]:
class FeatureDataset(Dataset):
    def __init__(self, fo_view, label_to_id: dict[str, int]):
        self.samples = []
        for s in fo_view.select_fields(
            ["siglip2_emb", "canonical_label", "sample_weight", "id"]
        ):
            emb = np.array(s.siglip2_emb, dtype=np.float32)
            label_id = label_to_id[s.canonical_label]
            weight = s.sample_weight if s.sample_weight is not None else 1.0

            # Using tuple or dict. Trainer expects dict.
            self.samples.append(
                {
                    "inputs_embeds": emb,
                    "labels": label_id,
                    "sample_weight": weight,
                    "sample_id": str(s.id),
                }
            )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


class LinearHeadModel(nn.Module):
    def __init__(self, input_dim=1152, num_labels=6):
        super().__init__()
        self.num_labels = num_labels
        self.classifier = nn.Linear(input_dim, num_labels)

    def forward(self, inputs_embeds, labels=None, sample_weight=None, sample_id=None):
        logits = self.classifier(inputs_embeds)
        # Loss will be computed by CustomTrainer
        return {"logits": logits}


class CustomFeatureTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        if class_weights is not None:
            self.class_weights = torch.tensor(class_weights, dtype=torch.float32)
        else:
            self.class_weights = None

    def compute_loss(
        self, model, inputs, return_outputs=False, num_items_in_batch=None
    ):
        labels = inputs.pop("labels")
        sample_weights = inputs.pop("sample_weight", None)
        inputs.pop("sample_id", None)

        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.class_weights is not None:
            class_weights = self.class_weights.to(logits.device)
        else:
            class_weights = None

        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights, reduction="none", label_smoothing=0.1
        )
        loss = loss_fct(logits, labels)

        if sample_weights is not None:
            loss = loss * sample_weights

        loss = loss.mean()
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred.predictions, eval_pred.label_ids
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.argmax(logits, axis=-1)

    return {
        "macro_f1": f1_score(labels, preds, average="macro"),
        "balanced_acc": balanced_accuracy_score(labels, preds),
        "accuracy": accuracy_score(labels, preds),
    }

[No output generated]

In [ ]:
def collate_fn(batch):
    return {
        "inputs_embeds": torch.tensor(
            np.stack([x["inputs_embeds"] for x in batch]), dtype=torch.float32
        ),
        "labels": torch.tensor([x["labels"] for x in batch], dtype=torch.long),
        "sample_weight": torch.tensor(
            [x["sample_weight"] for x in batch], dtype=torch.float32
        ),
        "sample_id": [x["sample_id"] for x in batch],
    }

[No output generated]

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

oof_preds_dict = {}

# MLflow setup
experiment_name = "Stage4_Baseline"
mlflow.set_experiment(experiment_name)

all_f1 = []
all_bal_acc = []

# Generate timestamp for unique run names
run_ts = datetime.now().strftime("%m%d-%H%M")

for fold in range(5):
    print(f"=== Training Fold {fold} ===")

    train_view = view.match(fo.ViewField("fold_id") != fold)
    val_view = view.match(fo.ViewField("fold_id") == fold)

    train_ds = FeatureDataset(train_view, label2id)
    val_ds = FeatureDataset(val_view, label2id)

    # Tính class weights cho fold này
    y_train = [train_ds[i]["labels"] for i in range(len(train_ds))]

    c_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
    # create a map from unique -> weights
    weight_map = dict(zip(np.unique(y_train), c_weights, strict=True))
    final_class_weights = [weight_map.get(i, 1.0) for i in range(len(labels_list))]

    model = LinearHeadModel(input_dim=1152, num_labels=len(labels_list))

    args = TrainingArguments(
        output_dir=f"./checkpoints/baseline_cv_fold_{fold}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=1e-3,
        auto_find_batch_size=True,
        num_train_epochs=5,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        report_to="mlflow",
        run_name=f"baseline-fold-{fold}-{run_ts}",
    )

    trainer = CustomFeatureTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
        class_weights=final_class_weights,
    )

    trainer.train()

    # Predict on validation subset
    val_outputs = trainer.predict(val_ds)
    all_f1.append(val_outputs.metrics["test_macro_f1"])
    all_bal_acc.append(val_outputs.metrics["test_balanced_acc"])

    logits = val_outputs.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()

    for i, item in enumerate(val_ds):
        sample_id = item["sample_id"]
        class_id = int(np.argmax(probs[i]))
        confidence = float(np.max(probs[i]))
        oof_preds_dict[sample_id] = {
            "label": id2label[class_id],
            "confidence": confidence,
            "logits": [float(x) for x in logits[i]],
        }

print("=== CV Finished ===")
print(f"Mean Macro F1: {np.mean(all_f1):.4f}")
print(f"Mean Balanced Acc: {np.mean(all_bal_acc):.4f}")

2026/03/14 03:09:19 INFO mlflow.tracking.fluent: Experiment with name 'Stage4_Baseline' does not exist. Creating a new experiment.


=== Training Fold 0 ===


---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
Cell In[23], line 31
     27 final_class_weights = [weight_map.get(i, 1.0) for i in range(len(labels_list))]
     29 model = LinearHeadModel(input_dim=1152, num_labels=len(labels_list))
---> 31 args = TrainingArguments(
     32     output_dir=f"./baseline_cv_fold_{fold}",
     33     evaluation_strategy="epoch",
     34     save_strategy="epoch",
     35     learning_rate=1e-3,
     36     auto_find_batch_size=True,
     37     num_train_epochs=5,
     38     load_best_model_at_end=True,
     39     metric_for_best_model="macro_f1",
     40     report_to="mlflow",
     41     run_name=f"baseline-fold-{fold}"
     42 )
     44 trainer = CustomFeatureTrainer(
     45     model=model,
     46     args=args,
   (...)     51     class_weights=final_class_weights
     52 )
     54 trainer.train()

TypeError: TrainingArguments.__init__() got a

In [ ]:
# RE-RUN FOLD 0 with eval_strategy
all_f1 = []
all_bal_acc = []

# Generate timestamp for unique run names
run_ts = datetime.now().strftime("%m%d-%H%M")

for fold in range(5):
    print(f"=== Training Fold {fold} ===")

    train_view = view.match(fo.ViewField("fold_id") != fold)
    val_view = view.match(fo.ViewField("fold_id") == fold)

    train_ds = FeatureDataset(train_view, label2id)
    val_ds = FeatureDataset(val_view, label2id)

    y_train = [train_ds[i]["labels"] for i in range(len(train_ds))]

    c_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
    weight_map = dict(zip(np.unique(y_train), c_weights, strict=True))
    final_class_weights = [weight_map.get(i, 1.0) for i in range(len(labels_list))]

    model = LinearHeadModel(input_dim=1152, num_labels=len(labels_list))

    args = TrainingArguments(
        output_dir=f"./checkpoints/baseline_cv_fold_{fold}",
        eval_strategy="epoch",  # Updated to eval_strategy
        save_strategy="epoch",
        learning_rate=1e-3,
        auto_find_batch_size=True,
        num_train_epochs=5,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        report_to="mlflow",
        run_name=f"baseline-fold-{fold}-{run_ts}",
    )

    # Needs to match class weight
    trainer = CustomFeatureTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
        class_weights=final_class_weights,
    )

    trainer.train()

    # Predict on validation subset
    val_outputs = trainer.predict(val_ds)
    all_f1.append(val_outputs.metrics["test_macro_f1"])
    all_bal_acc.append(val_outputs.metrics["test_balanced_acc"])

    logits = val_outputs.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()

    for i, item in enumerate(val_ds):
        sample_id = item["sample_id"]
        class_id = int(np.argmax(probs[i]))
        confidence = float(np.max(probs[i]))
        oof_preds_dict[sample_id] = {
            "label": id2label[class_id],
            "confidence": confidence,
            "logits": [float(x) for x in logits[i]],
        }

print("=== CV Finished ===")
print(f"Mean Macro F1: {np.mean(all_f1):.4f}")
print(f"Mean Balanced Acc: {np.mean(all_bal_acc):.4f}")

=== Training Fold 0 ===


/Users/anh.pham/Personal/zalo-photos-assignment-2026/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


<IPython.core.display.HTML object>

/Users/anh.pham/Personal/zalo-photos-assignment-2026/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


/Users/anh.pham/Personal/zalo-photos-assignment-2026/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


...

/Users/anh.pham/Personal/zalo-photos-assignment-2026/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


🏃 View run baseline-fold-4 at: https://dagshub.com/cykanp/zalo-photos-assignment-2026.mlflow/#/experiments/0/runs/d82abc06fc7a4f5f959224dd5c422166
🧪 View experiment at: https://dagshub.com/cykanp/zalo-photos-assignment-2026.mlflow/#/experiments/0


/Users/anh.pham/Personal/zalo-photos-assignment-2026/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


<IPython.core.display.HTML object>

=== CV Finished ===
Mean Macro F1: 0.8425
Mean Balanced Acc: 0.8924


In [ ]:
# Lưu raw OOF logits & predictions vào dataset (cho local)
for sample_id, preds in oof_preds_dict.items():
    sample = dataset[sample_id]
    sample["pred_head"] = fo.Classification(
        label=preds["label"], confidence=preds["confidence"], logits=preds["logits"]
    )
    sample.save()

[No output generated]

In [ ]:
import fiftyone as fo

# --- EXPORT DATA FOR CUDA MACHINE ---
# Khác với raw dataset, ta dùng FiftyOneDataset format
# để giữ nguyên sample fields đặc thù (embeddings, weights).
export_dir = "data/z_photos-gold-train-export"
print(f"Exporting FiftyOne dataset to {export_dir}...")

dataset = fo.load_dataset("z_photos-gold-train")
dataset.export(
    export_dir=export_dir,
    dataset_type=fo.types.FiftyOneDataset,
    export_media=False,  # Chỉ export metadata để transfer (images gốc đã có trên cuda)
    overwrite=True,
)
print(
    "Export complete!",
    "Copy `data/z_photos-gold-train-export/` to other machine",
    "and run `z_photos/train.py`.",
)

Exporting FiftyOne dataset to data/z_photos-gold-train-export...
Exporting samples...



   0% ||-------------------|   0/321 [9.6us elapsed, ? remaining, ? docs/s] 


  55% |██████████/---------| 175/321 [100.2ms elapsed, 83.6ms remaining, 1.7K docs/s] 


 100% |████████████████████| 321/321 [173.2ms elapsed, 0s remaining, 1.9K docs/s]     


Export complete! You can copy `data/z_photos-gold-train-export/` to your CUDA machine and run `src/z_photos/train.py`.
